In [1]:
"""
DO NOT USE THIS FOR TRAINING, ONLY FOR PROTOTYPING
"""

import torch
from config import TRAINING_DEFAULTS
from torch import nn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from data.dataloader import SionnaMultiSceneDataset, SnapshotDataset
from torch_geometric.loader import DataLoader

dataset = SionnaMultiSceneDataset(
    filepaths=[
        "./data/multi_scene_ds/munich_train.h5",
        "./data/multi_scene_ds/florence_train.h5"
    ])

snapshot_dataset = SnapshotDataset(dataset)

dataloader = DataLoader(
    snapshot_dataset,
    batch_size=TRAINING_DEFAULTS["BATCH_SIZE"],
    shuffle=True,
    num_workers=0,
)

print(f"num_batches: {len(dataloader)}")
print(f"num_graphs: {len(snapshot_dataset)}")

/home/crk/Projects/THWS/Project Module/gnss-jammer-localization/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


num_batches: 7500
num_graphs: 60000


In [5]:
from model import GNN

model = GNN()
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAINING_DEFAULTS["LEARNING_RATE"],
    weight_decay=TRAINING_DEFAULTS["WEIGHT_DECAY"],
)
criterion = nn.MSELoss()

In [6]:
batch = next(iter(dataloader))
print(batch.y_pos.shape)   # [8, 3]
print(batch.wcl_pos.shape) # [8, 3]

torch.Size([8, 3])
torch.Size([8, 3])


In [7]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                        Param #
GNN                                           --
├─GAT: 1-1                                    --
│    └─Dropout: 2-1                           --
│    └─ReLU: 2-2                              --
│    └─ModuleList: 2-3                        --
│    │    └─GATConv: 3-1                      1,280
│    │    └─GATConv: 3-2                      16,768
│    │    └─GATConv: 3-3                      16,768
│    │    └─GATConv: 3-4                      16,768
│    │    └─GATConv: 3-5                      16,768
│    │    └─GATConv: 3-6                      16,768
│    │    └─GATConv: 3-7                      16,768
│    │    └─GATConv: 3-8                      66,688
│    └─ModuleList: 2-4                        --
│    │    └─Identity: 3-9                     --
│    │    └─Identity: 3-10                    --
│    │    └─Identity: 3-11                    --
│    │    └─Identity: 3-12                    --
│    │    └─Identity: 3-13       

In [ ]:
# Training loop
for epoch in range(10):
    for batch in dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()
gnn_prediction, final_prediction, weight = model(batch)

        loss_gnn = criterion(gnn_prediction, batch.y_pos)
        loss_final = criterion(final_prediction, batch.y_pos)
        loss = (loss_gnn + loss_final) / 2

        
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item()}")

batch Loss: 7468.64306640625
batch Loss: 3384.2705078125
batch Loss: 5208.17138671875
batch Loss: 5702.62646484375
batch Loss: 4273.42822265625
batch Loss: 3709.953857421875
batch Loss: 2549.393798828125
batch Loss: 1928.863037109375
batch Loss: 3302.348388671875
batch Loss: 2122.69091796875
batch Loss: 5668.42041015625
batch Loss: 2110.162841796875
batch Loss: 2844.195556640625
batch Loss: 1463.85498046875
batch Loss: 2770.75732421875
batch Loss: 1224.480224609375
batch Loss: 2465.716552734375
batch Loss: 1956.8367919921875
batch Loss: 3443.76220703125
batch Loss: 2832.66845703125
batch Loss: 1948.8013916015625
batch Loss: 973.9451904296875
batch Loss: 1210.68408203125
batch Loss: 1220.74853515625
batch Loss: 1615.2337646484375
batch Loss: 1304.1856689453125
batch Loss: 1739.884033203125
batch Loss: 1199.4178466796875
batch Loss: 1723.2919921875
batch Loss: 1455.442138671875
batch Loss: 1410.3223876953125
batch Loss: 1495.25634765625
batch Loss: 1049.075439453125
batch Loss: 922.42596

KeyboardInterrupt: 